#  **Assignment_Classification_without_code - Bartosz Kołacz.**

### Today we are going to perform the simple classification of the amazon reviews' sentiment.

### Please, download the dataset amazon_baby.csv.

In [84]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import string
from sklearn.linear_model import LogisticRegression

def remove_punctuation(text):
    import string
    translator = str.maketrans('', '', string.punctuation)
    return text.translate(translator)

baby_df = pd.read_csv('amazon_baby.csv')
baby_df.head()

,name,review,rating
0,Planetwise Flannel Wipes,"These flannel wipes are OK, but in my opinion ...",3
1,Planetwise Wipe Pouch,it came early and was not disappointed. i love...,5
2,Annas Dream Full Quilt with 2 Shams,Very soft and comfortable and warmer than it l...,5
3,Stop Pacifier Sucking without tears with Thumb...,This is a product well worth the purchase. I ...,5
4,Stop Pacifier Sucking without tears with Thumb...,All of my kids have cried non-stop when I trie...,5


I found interesting observation. When I opened notebook and uploaded amazon_baby.csv for the first time, data was succesfuly loaded. When I opened the nootebook later and again uploaded .csv Google Colab returned me error with reading .csv. I found out that, the file was not fully read while I tried to process it, and that was the reason of the corruption.

## Exercise 1 (data preparation)
a) Remove punctuation from reviews using the given function.   
b) Replace all missing (nan) revies with empty "" string.  
c) Drop all the entries with rating = 3, as they have neutral sentiment.   
d) Set all positive ($\geq$4) ratings to 1 and negative($\leq$2) to -1.


In [85]:
#b)
baby_df["review"] = baby_df["review"].fillna("")

#short test:
baby_df["review"][38] == baby_df["review"][38]

True

In [86]:
#a)
baby_df["review"] = baby_df["review"].apply(remove_punctuation)

#short test:
baby_df["review"][4] == 'All of my kids have cried nonstop when I tried to ween them off their pacifier until I found Thumbuddy To Loves Binky Fairy Puppet  It is an easy way to work with your kids to allow them to understand where their pacifier is going and help them part from itThis is a must buy book and a great gift for expecting parents  You will save them soo many headachesThanks for this book  You all rock'
remove_punctuation(baby_df["review"][4]) == 'All of my kids have cried nonstop when I tried to ween them off their pacifier until I found Thumbuddy To Loves Binky Fairy Puppet  It is an easy way to work with your kids to allow them to understand where their pacifier is going and help them part from itThis is a must buy book and a great gift for expecting parents  You will save them soo many headachesThanks for this book  You all rock'

True

In [87]:
#c)
baby_df = baby_df[baby_df["rating"] != 3].copy()

#short test:
sum(baby_df["rating"] == 3)

0

In [88]:
#d)
baby_df["rating"] = baby_df["rating"].replace({1: -1, 2: -1, 4: 1, 5: 1})

#short test:
sum(baby_df["rating"]**2 != 1)

0

Annotation: I began with point b) to avoid transformaing of NaN. When I tried to do it in the indicated way, I received error: Google Colab was not able to process and remove punctation from NaN's so I changed the order of the instruction, and first of all I removed NaN's from columns including it and then i did point a). Then I removed reviews with rating equal 3, and then i ranged rating of other opinions.

## CountVectorizer
In order to analyze strings, we need to assign them numerical values. We will use one of the simplest string representation, which transforms strings into the $n$ dimensional vectors. The number of dimensions will be the size of our dictionary, and then the values of the vector will represent the number of appereances of the given word in the sentence.

In [89]:
from sklearn.feature_extraction.text import CountVectorizer

vectorizer = CountVectorizer()
reviews_train_example = ["We like apples",
                   "We hate oranges",
                   "I adore bananas",
                   "We like like apples and oranges",
                   "They dislike bananas"]

X_train_example = vectorizer.fit_transform(reviews_train_example)

print(vectorizer.get_feature_names_out())
print(X_train_example.todense())



['adore' 'and' 'apples' 'bananas' 'dislike' 'hate' 'like' 'oranges' 'they'
 'we']
[[0 0 1 0 0 0 1 0 0 1]
 [0 0 0 0 0 1 0 1 0 1]
 [1 0 0 1 0 0 0 0 0 0]
 [0 1 1 0 0 0 2 1 0 1]
 [0 0 0 1 1 0 0 0 1 0]]


In [90]:
reviews_test_example = ["They like bananas",
                   "We hate oranges bananas and apples",
                   "We love bananas"] #New word!

X_test_example = vectorizer.transform(reviews_test_example)

print(vectorizer.get_feature_names_out())
print(X_test_example.todense())

['adore' 'and' 'apples' 'bananas' 'dislike' 'hate' 'like' 'oranges' 'they'
 'we']
[[0 0 0 1 0 0 1 0 1 0]
 [0 1 1 1 0 1 0 1 0 1]
 [0 0 0 1 0 0 0 0 0 1]]


We should acknowledge few facts. Firstly, CountVectorizer does not take order into account. Secondly, it ignores one-letter words (this can be changed during initialization). Finally, for test values, CountVectorizer ignores words which are not in it's dictionary.

## Exercise 2
a) Split dataset into training and test sets.     
b) Transform reviews into vectors using CountVectorizer.

In [91]:
#a)
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(baby_df['review'], baby_df['rating'], test_size=0.3, random_state=42)

In [92]:
#b)
vectorizer = CountVectorizer()
X_train_model = vectorizer.fit_transform(X_train)
X_test_model = vectorizer.transform(X_test)

print(X_train_model.shape, X_test_model.shape)

(116726, 111574) (50026, 111574)


I splited data into two sets, train and test. Proportion 70/30 and random_state = 42. Then I vectorized them.

## Exercise 3
a) Train LogisticRegression model on training data (reviews processed with CountVectorizer, ratings as they were).   
b) Print 10 most positive and 10 most negative words.

In [93]:
#a)
model = LogisticRegression(max_iter=1000)
model.fit(X_train_model, y_train)

LogisticRegression(max_iter=1000)

In [94]:
#b)

words = vectorizer.get_feature_names_out()
coef = model.coef_.flatten()

top_positive_words = [words[i] for i in coef.argsort()[-10:][::-1]]
print("Top 10 POSITIVE words: " + str(top_positive_words))

top_negative_words = [words[i] for i in coef.argsort()[:10]]
print("Top 10 NEGATIVE words: " + str(top_negative_words))

#hint: model.coef_, vectorizer.get_feature_names()

Top 10 POSITIVE words: ['wonderfully', 'con', 'lifesaver', 'ply', 'minor', 'rich', 'saves', 'perfect', 'relax', 'glad']
Top 10 NEGATIVE words: ['dissapointed', 'worthless', 'worst', 'useless', 'disappointing', 'poorly', 'nope', 'disappointment', 'disappointed', 'poor']


I found out interesting fact that in list of the positive words model found: 'ply', 'minor' and 'con' as postive words. Meaning of them does not suggest they are positive. The reason is the fact that they are correlated to positivie opinions, and occurs in them a lot. Meaningly, they are not positive, but statisticaly, yes they are.

For the negative words list I can see the results are focused around 'disappointment'.

## Exercise 4
a) Predict the sentiment of test data reviews.   
b) Predict the sentiment of test data reviews in terms of probability.   
c) Find five most positive and most negative reviews.   
d) Calculate the accuracy of predictions.

In [95]:
#a)
y_prediction = model.predict(X_test_model)
print(y_prediction[:10])

[-1 -1 -1  1  1  1  1  1  1 -1]


In [96]:
#b)
y_prediction_proba = model.predict_proba(X_test_model)
print(y_prediction_proba[:10])

#hint: model.predict_proba()

[[5.19843627e-01 4.80156373e-01]
 [8.22368875e-01 1.77631125e-01]
 [9.90422573e-01 9.57742726e-03]
 [9.32516665e-04 9.99067483e-01]
 [1.56240237e-03 9.98437598e-01]
 [5.62468897e-04 9.99437531e-01]
 [7.57703191e-03 9.92422968e-01]
 [7.95433735e-02 9.20456626e-01]
 [2.43632075e-02 9.75636793e-01]
 [8.88727410e-01 1.11272590e-01]]


In [97]:
#c)
order = np.argsort(y_prediction_proba[:, 1])
print("Top 5 NEGATIVE reviews:")
print(X_test.iloc[order[:5]].values)

print("Top 5 POSITIVE reviews:")
print(X_test.iloc[order[-5:]].values)

Top 5 NEGATIVE reviews:
['My disappointment with this product prompted me to write to the manufacturer  Below is my correspondenceIm writing you this email to express my deep dissatisfaction with a recently purchased Graco productMy wife and I are new parents and as expected the Graco brand was quite heavily represented on many items that expectant parents needAmong other things we purchased a Graco car seat extra seat base and a Pack and PlayAt this point our daughter is only a week old the car seat has seen limited use but the Pack and Play has already been put through some paces and it is this product that I am writing to you aboutThe model we purchased was the Pack n Play Dempsey  The main reasons we chose this model were the designcolor and the fact that it had a changing table  The additional storage seemed to be a nice feature thought not something that we will probably use at this point and as it disappears when not used as a bassinet it has limited usefulnessI have quite a few

In [98]:
#d)
from sklearn.metrics import accuracy_score

accuracy = accuracy_score(y_test, y_prediction)
print("Accuracy:", accuracy)

Accuracy: 0.9304361731899412


My train model succesfuly recognised and classified 93% of the test reviews. That result is highly satisfying.

## Exercise 5
In this exercise we will limit the dictionary of CountVectorizer to the set of significant words, defined below.


a) Redo exercises 2-5 using limited dictionary.   
b) Check the impact of all the words from the dictionary.   
c) Compare accuracy of predictions and the time of evaluation.

In [99]:
significant_words = ['love','great','easy','old','little','perfect','loves','well','able','car','broke','less','even','waste','disappointed','work','product','money','would','return']

In [100]:
#a)

# ex 2: data split

from sklearn.model_selection import train_test_split

vectorizer_5 = CountVectorizer(vocabulary = significant_words)
X_train_model_5 = vectorizer_5.fit_transform(X_train)
X_test_model_5 = vectorizer_5.transform(X_test)


In [102]:
# ex 3:

model_5 = LogisticRegression(max_iter=1000)
model_5.fit(X_train_model_5, y_train)

words_5 = vectorizer_5.get_feature_names_out()
coef_5 = model_5.coef_.flatten()

top_positive_words = [words_5[i] for i in coef_5.argsort()[-10:][::-1]]
print("Top 10 POSITIVE words: " + str(top_positive_words))

top_negative_words = [words_5[i] for i in coef_5.argsort()[:10]]
print("Top 10 NEGATIVE words: " + str(top_negative_words))

Top 10 POSITIVE words: ['loves', 'perfect', 'love', 'easy', 'great', 'little', 'well', 'able', 'car', 'old']
Top 10 NEGATIVE words: ['disappointed', 'return', 'waste', 'broke', 'money', 'work', 'even', 'would', 'product', 'less']


In [103]:
# ex 4

y_prediction_5 = model_5.predict(X_test_model_5)
y_prediction_proba_5 = model_5.predict_proba(X_test_model_5)

order = np.argsort(y_prediction_proba_5[:, 1])
print("Top 5 NEGATIVE reviews:")
print(X_test.iloc[order[:5]].values)

print("Top 5 POSITIVE reviews:")
print(X_test.iloc[order[-5:]].values)

Top 5 NEGATIVE reviews:
['Looks really cute however the cloth smells funny and gave my baby some rash after 15 mins wearingRequested a return need to pay 25 restocking fee plus shipping25 order maybe 5 back eventuallyAlso the return must be brand new condition Really UnbelievableDo yourself a favor and See the return policy by yourself before making an orderHelloBelow is return instructions this item must not be worn or used or the return will not be refundedThank you for ordering from us Here is your RMA as well as return instructions All returns must be in brand new original condition Please read our return policy Return policy link is provided for you in this emailAll refunds issued are determined by the outcome of inspections Any return of costumes used worn signs of pet hair human hair or washed will void all refund All boxes for shoes NEED to be unmarked and in original condition ALL returns NEED to be in original packaging to qualify for an authorized returnPlease note If you ar

In [104]:
#b)
impact = pd.DataFrame({"word": significant_words, "coef": model_5.coef_[0]})
print(impact.sort_values("coef"))

            word      coef
14  disappointed -2.407815
19        return -2.099910
13         waste -1.978353
10         broke -1.651832
17         money -0.945372
15          work -0.624781
12          even -0.504354
18         would -0.347826
16       product -0.311687
11          less -0.200767
3            old  0.059585
9            car  0.085912
8           able  0.193161
7           well  0.497090
4         little  0.519760
1          great  0.914435
2           easy  1.180044
0           love  1.347673
5        perfect  1.512540
6          loves  1.704436


I did this subsuction in the same way as previous excercises. The most magnificant impact on reviews classification had: loves (1.70) and disappointed (-2.40).

In [105]:
#c)
from sklearn.metrics import accuracy_score

print("Original accuracy: \t    0.9304361731899412")

accuracy = accuracy_score(y_test, y_prediction_5)
print("Significant_words accuracy:", accuracy)

print("\nOriginal time:")
%time model.fit(X_train_model, y_train)
print("\nSignificant_words time:")
%time model_5.fit(X_train_model_5, y_train)
#hint: %time, %timeit

print(X_train_model.shape, X_train_model_5.shape)

Original accuracy: 	    0.9304361731899412
Significant_words accuracy: 0.8674689161635949

Original time:
CPU times: user 1min 50s, sys: 177 ms, total: 1min 50s
Wall time: 1min 3s

Significant_words time:
CPU times: user 422 ms, sys: 10 µs, total: 422 ms
Wall time: 232 ms
(116726, 111574) (116726, 20)


Using the limited words vocabulary magnificantly fasten the classification, from almost 2 minutes to only fraction of the second. However, with the usage of limited vocabulary accuracy dropped for almost 7%.